# Part 3 – Market Intelligence Dashboard
Put `curated_market_data.csv` in the same folder as this notebook, then click **Run All**.
(In Colab: Runtime → Run all, and pick the CSV when asked.)

In [ ]:
# =============================================================
# Part 3 – Market Intelligence Dashboard (S&P 500 + Fed Funds Rate)
# Input : curated_market_data.csv  (Bhagya's cleaned dataset, Part 2)
# Output: dashboard.png  + printed "what to watch" summary
#
# HOW TO RUN IN GOOGLE COLAB
#   1. Upload curated_market_data.csv (folder icon on the left -> upload)
#   2. Paste each "# %%" block into its own cell and run top to bottom
# =============================================================

In [ ]:
# %% 1. Setup and load data
import os, tempfile
# Give matplotlib a folder it can write to (fixes "writable cache directory" error on some Macs)
os.environ["MPLCONFIGDIR"] = os.path.join(tempfile.gettempdir(), "mpl_cache")
os.makedirs(os.environ["MPLCONFIGDIR"], exist_ok=True)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

# --- Find the data file (or ask you to upload it) ---------------
import glob
matches = [f for f in glob.glob("**/*.csv", recursive=True) if "curated" in f.lower()]
if not matches:
    try:
        from google.colab import files
        print("Data file not found. Click 'Choose files' below and pick curated_market_data.csv")
        uploaded = files.upload()
        matches = [f for f in uploaded if f.lower().endswith(".csv")]
    except ImportError:
        pass
if not matches:
    raise FileNotFoundError("Upload curated_market_data.csv, then run this cell again.")
CSV_FILE = matches[0]
print("Using data file:", CSV_FILE)
WINDOW_DAYS = 90            # dashboard window (calendar days)

# RAG thresholds on ANNUALISED 14-day volatility (%)
GREEN_MAX = 15              # below 15% = calm (below long-run S&P average of ~15-20%)
AMBER_MAX = 25              # 15-25% = elevated, monitor;  above 25% = stress, escalate

df = pd.read_csv(CSV_FILE)
# utc=True copes with dates with or without a time-zone suffix
df["Date"] = pd.to_datetime(df["Date"], utc=True).dt.tz_localize(None).dt.normalize()
df = df.sort_values("Date").reset_index(drop=True)
print(df.shape, df["Date"].min().date(), "->", df["Date"].max().date())
print(df.columns.tolist())

In [ ]:
# %% 2. Metrics needed by the dashboard
# 14-day rolling average of the closing price (not in the Part 2 file, so added here)
if "Rolling_Avg_14" not in df.columns:
    df["Rolling_Avg_14"] = df["Close"].rolling(14).mean()

# Independent check of Bhagya's Volatility_14: std-dev of daily returns over 14 days
check_daily = df["Close"].pct_change().rolling(14).std()

# Work out which units Volatility_14 is in by comparing with our own calculation
ratio = (df["Volatility_14"] / check_daily).median()
units = {1: "daily decimal", 100: "daily %",
         np.sqrt(252): "annualised decimal", np.sqrt(252) * 100: "annualised %"}
best = min(units, key=lambda k: abs(np.log(ratio / k)))
print(f"Volatility_14 looks like: {units[best]} (ratio to check = {ratio:.2f})")

# Convert to annualised % so the thresholds above make sense
df["Vol_Annual_Pct"] = df["Volatility_14"] / best * np.sqrt(252) * 100

# Sanity check: our figure and Bhagya's should agree closely
diff = (df["Vol_Annual_Pct"] - check_daily * np.sqrt(252) * 100).abs().max()
print(f"Max difference vs independent calculation: {diff:.4f} percentage points")

In [ ]:
# %% 3. RAG signal
def rag(vol):
    if pd.isna(vol):
        return "Grey", "Not enough data"
    if vol < GREEN_MAX:
        return "Green", "Calm – volatility below normal"
    if vol < AMBER_MAX:
        return "Amber", "Elevated – monitor closely"
    return "Red", "High stress – escalate to management"

COLOURS = {"Green": "#2e7d32", "Amber": "#f9a825", "Red": "#c62828", "Grey": "#9e9e9e"}

In [ ]:
# %% 4. Keep the last 90 calendar days and work out the story
end = df["Date"].max()
d = df[df["Date"] > end - pd.Timedelta(days=WINDOW_DAYS)].copy().reset_index(drop=True)

latest = d.iloc[-1]
change_90 = (latest["Close"] / d.iloc[0]["Close"] - 1) * 100
status, status_text = rag(latest["Vol_Annual_Pct"])

# What the reader should DO for each status (Week 12: every alert needs an action)
ACTIONS = {"Green": "No action needed –\nroutine weekly check.",
           "Amber": "Watch daily; review limits\nif it lasts 5+ days.",
           "Red":   "Escalate to head of desk\nthe same day.",
           "Grey":  "Check the data feed."}

# How many trading days has the signal been at this status?
statuses = d["Vol_Annual_Pct"].apply(lambda v: rag(v)[0])
streak = (statuses[::-1] == status).cumprod().sum()

above = latest["Close"] > latest["Rolling_Avg_14"]

# Did the Fed change rates in the window?
d["DFF_change"] = d["DFF"].diff()
moves = d[d["DFF_change"].abs() > 0.10]

In [ ]:
# %% 5. Build the dashboard (Week 10 rules: takeaway titles, grey + one accent,
#       no gridlines, no legends - label lines directly, left-aligned text)
GREY, DARK, GOLD = "#9a9a9a", "#3a3a3a", "#c8961e"
plt.rcParams.update({"font.family": "DejaVu Sans", "axes.spines.top": False,
                     "axes.spines.right": False, "axes.edgecolor": GREY,
                     "xtick.color": DARK, "ytick.color": DARK})

fig = plt.figure(figsize=(13, 14))
gs = fig.add_gridspec(4, 4, height_ratios=[0.55, 1.4, 1.2, 1.0],
                      hspace=0.5, wspace=0.3, top=0.91, bottom=0.08)

# Takeaway title: states the insight, and updates itself when the data changes
headline = (f"The S&P 500 is {'up' if change_90 >= 0 else 'down'} {abs(change_90):.1f}% "
            f"over 90 days, and the risk signal is {status.upper()}")
fig.text(0.06, 0.955, headline, fontsize=18, fontweight="bold", color=DARK)
fig.text(0.06, 0.93, f"Market Intelligence Dashboard  |  last 90 days to {end:%d %b %Y}  |  "
         "Sources: Yahoo Finance (^GSPC), FRED (DFF)", fontsize=10, color=GREY)

def label_end(ax, x, y, text, colour):
    """Direct label at the end of a line, instead of a legend."""
    ax.annotate(text, (x, y), xytext=(6, 0), textcoords="offset points",
                color=colour, fontsize=10, fontweight="bold", va="center")

def clean(ax):
    ax.grid(False)
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%d %b"))
    ax.set_xlim(d["Date"].min(), end + pd.Timedelta(days=11))   # room for end labels

# --- KPI tiles: grey context, one coloured status tile --------
def tile(ax, title, value, sub="", colour="#f3f3f3", text=DARK):
    ax.set_facecolor(colour); ax.set_xticks([]); ax.set_yticks([])
    for s in ax.spines.values(): s.set_visible(False)
    ax.text(0.06, 0.78, title, fontsize=10, color=text, transform=ax.transAxes)
    ax.text(0.06, 0.40, value, fontsize=20, fontweight="bold", color=text, transform=ax.transAxes)
    ax.text(0.06, 0.08, sub, fontsize=9, color=text, transform=ax.transAxes, va="bottom")

tile(fig.add_subplot(gs[0, 0]), "S&P 500 latest close", f"{latest['Close']:,.0f}", f"{end:%d %b %Y}")
tile(fig.add_subplot(gs[0, 1]), "Change over 90 days", f"{change_90:+.1f}%",
     f"since {d.iloc[0]['Date']:%d %b}")
tile(fig.add_subplot(gs[0, 2]), "US Fed Funds Rate", f"{latest['DFF']:.2f}%",
     f"{len(moves)} change(s) in 90 days")
tile(fig.add_subplot(gs[0, 3]), f"Risk signal: {status.upper()}",
     f"{latest['Vol_Annual_Pct']:.1f}%", ACTIONS[status],
     colour=COLOURS[status], text="white" if status in ("Green", "Red") else DARK)

# --- Chart 1: price + 14-day rolling average (CONTEXT) --------
ax1 = fig.add_subplot(gs[1, :])
ax1.plot(d["Date"], d["Close"], color=GREY, lw=1.6)
ax1.plot(d["Date"], d["Rolling_Avg_14"], color=GOLD, lw=2.8)
label_end(ax1, end, latest["Close"], "Daily close", GREY)
label_end(ax1, end, latest["Rolling_Avg_14"], "14-day average", GOLD)
ax1.set_title(f"1. The index is {'ABOVE' if above else 'BELOW'} its 14-day average, so "
              f"short-term momentum is {'positive' if above else 'weakening'}",
              loc="left", fontsize=12, fontweight="bold", color=DARK)
ax1.set_ylabel("Index points", color=GREY)
clean(ax1)

# --- Chart 2: S&P 500 vs Fed Funds Rate (WHAT CHANGED) ---------
ax2 = fig.add_subplot(gs[2, :])
ax2.plot(d["Date"], d["Close"], color=GREY, lw=1.6)
ax2.set_ylabel("S&P 500 (index points)", color=GREY)
ax2b = ax2.twinx()
ax2b.step(d["Date"], d["DFF"], where="post", color=GOLD, lw=2.8)
ax2b.set_ylabel("Fed Funds Rate (%)", color=GOLD)
pad = max(0.25, (d["DFF"].max() - d["DFF"].min()) * 0.5)
ax2b.set_ylim(d["DFF"].min() - pad, d["DFF"].max() + pad)
ax2b.spines["right"].set_visible(True); ax2b.spines["top"].set_visible(False)
label_end(ax2, end, latest["Close"], "S&P 500", GREY)
label_end(ax2b, end, latest["DFF"], "Fed rate", GOLD)

if len(moves):
    m = moves.iloc[-1]
    after = d[d["Date"] >= m["Date"]].head(10)                      # ~2 trading weeks
    move_after = (after.iloc[-1]["Close"] / after.iloc[0]["Close"] - 1) * 100
    verb = "cut" if m["DFF_change"] < 0 else "raised"
    ax2.axvline(m["Date"], color=DARK, ls=":", lw=1)                # connection
    ax2.annotate(f"Fed {verb} rates\n{m['Date']:%d %b}", (m["Date"], ax2.get_ylim()[1]),
                 xytext=(5, -4), textcoords="offset points", va="top", fontsize=9, color=DARK)
    title2 = (f"2. The Fed {verb} rates by {abs(m['DFF_change']):.2f} pts on {m['Date']:%d %b}; "
              f"the S&P 500 moved {move_after:+.1f}% over the next two weeks")
else:
    title2 = (f"2. Rates held at {latest['DFF']:.2f}% all period, so the market's "
              f"{change_90:+.1f}% move was driven by other factors")
ax2.set_title(title2, loc="left", fontsize=12, fontweight="bold", color=DARK)
clean(ax2)

# --- Chart 3: volatility against the RAG bands (WHAT TO WATCH) -
ax3 = fig.add_subplot(gs[3, :])
top = max(AMBER_MAX + 10, d["Vol_Annual_Pct"].max() * 1.15)
for lo, hi, c, name in [(0, GREEN_MAX, "Green", "GREEN – calm"),
                        (GREEN_MAX, AMBER_MAX, "Amber", "AMBER – monitor"),
                        (AMBER_MAX, top, "Red", "RED – escalate")]:
    ax3.axhspan(lo, hi, color=COLOURS[c], alpha=0.12, lw=0)        # enclosure
    ax3.text(1.01, (lo + hi) / 2, name, fontsize=9, color=COLOURS[c], va="center",
             fontweight="bold", transform=ax3.get_yaxis_transform())
ax3.plot(d["Date"], d["Vol_Annual_Pct"], color=DARK, lw=2)
ax3.scatter([end], [latest["Vol_Annual_Pct"]], color=COLOURS[status], s=60, zorder=3)
label_end(ax3, end, latest["Vol_Annual_Pct"], f"{latest['Vol_Annual_Pct']:.1f}% today", DARK)
ax3.set_ylim(0, top); ax3.set_ylabel("14-day volatility (%, annualised)", color=GREY)
ax3.set_title(f"3. {status.upper()}: volatility has been in the {status.lower()} zone "
              f"for {streak} trading day(s)", loc="left", fontsize=12,
              fontweight="bold", color=COLOURS[status] if status != "Amber" else DARK)
clean(ax3)

# Data note (Week 11 workshop: say what was cleaned or assumed)
fig.text(0.06, 0.035,
         "Data notes: S&P 500 dates are US trading days. The Fed rate is published daily (incl. weekends) "
         "and the latest value is carried forward to each trading day.\n"
         f"Volatility = standard deviation of daily returns over 14 trading days, annualised (x sqrt 252). "
         f"Thresholds: Green < {GREEN_MAX}%, Amber {GREEN_MAX}-{AMBER_MAX}%, Red > {AMBER_MAX}%.",
         fontsize=8.5, color=GREY)

plt.savefig("dashboard_after.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# %% 6. "What to watch" summary for management (plain English)
days_red = (d["Vol_Annual_Pct"] >= AMBER_MAX).sum()
print("WHAT MANAGEMENT SHOULD WATCH")
print(f"- Trend: the S&P 500 is {'above' if above else 'below'} its 14-day average and "
      f"{'up' if change_90 >= 0 else 'down'} {abs(change_90):.1f}% over 90 days.")
print(f"- Rates: the Fed Funds Rate is {latest['DFF']:.2f}% with {len(moves)} change(s) in the window.")
print(f"- Risk: volatility is {latest['Vol_Annual_Pct']:.1f}% ({status}) for {streak} day(s); "
      f"{days_red} day(s) in the Red zone over the period. Action: {ACTIONS[status]}")